In [31]:
# Imports
import os
import re
import ast
import numpy as np
import pandas as pd

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

In [4]:
# Paths
DATA_TSV   = "../data/raw/dontpatronizeme_pcl.tsv"
TRAIN_CSV  = "../data/raw/train_semeval_parids-labels.csv"
DEV_CSV    = "../data/raw/dev_semeval_parids-labels.csv"
TEST_TSV   = "../data/raw/task4_test.tsv"

os.makedirs("../BestModel", exist_ok=True)

In [9]:
# Load main TSV 
rows = []
bad_lines = 0
with open(DATA_TSV, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) == 6:
            rows.append(parts)
        else:
            bad_lines += 1

print("Loaded rows:", len(rows), "Bad lines skipped:", bad_lines)

base = pd.DataFrame(rows, columns=["row_id","par_id","keyword","country","paragraph","label_0_4"])
base["row_id"] = pd.to_numeric(base["row_id"], errors="coerce")
base = base.dropna(subset=["row_id"]).copy()
base["row_id"] = base["row_id"].astype(int)

base["par_id"] = base["par_id"].astype(str)
base["par_id_noat"] = base["par_id"].str.replace("^@@", "", regex=True)

base = base[["row_id", "par_id", "par_id_noat", "paragraph"]].copy()
base.head()

Loaded rows: 10469 Bad lines skipped: 4


,row_id,par_id,par_id_noat,paragraph
0,1,@@24942188,24942188,"We 're living in times of absolute insanity , ..."
1,2,@@21968160,21968160,"In Libya today , there are countless number of..."
2,3,@@16584954,16584954,"""White House press secretary Sean Spicer said ..."
3,4,@@7811231,7811231,Council customers only signs would be displaye...
4,5,@@1494111,1494111,""""""" Just like we received migrants fleeing El ..."


In [28]:
# Load official train/dev splits 
train_ids = pd.read_csv(TRAIN_CSV)
dev_ids   = pd.read_csv(DEV_CSV)

print("TRAIN raw columns:", train_ids.columns.tolist())
print("DEV raw columns:", dev_ids.columns.tolist())

# Keep only first 2 columns (ID + label), ignore any extra columns
train_ids = train_ids.iloc[:, :2].copy()
dev_ids   = dev_ids.iloc[:, :2].copy()

# Force names
train_ids.columns = ["row_id", "label"]
dev_ids.columns   = ["row_id", "label"]

# Convert row_id to int
train_ids["row_id"] = pd.to_numeric(train_ids["row_id"], errors="coerce")
dev_ids["row_id"]   = pd.to_numeric(dev_ids["row_id"], errors="coerce")

train_ids = train_ids.dropna(subset=["row_id"]).copy()
dev_ids   = dev_ids.dropna(subset=["row_id"]).copy()

train_ids["row_id"] = train_ids["row_id"].astype(int)
dev_ids["row_id"]   = dev_ids["row_id"].astype(int)

print("TRAIN cleaned columns:", train_ids.columns.tolist())
print("DEV cleaned columns:", dev_ids.columns.tolist())

train_ids.head()

TRAIN raw columns: ['par_id', 'label']
DEV raw columns: ['par_id', 'label']
TRAIN cleaned columns: ['row_id', 'label']
DEV cleaned columns: ['row_id', 'label']


,row_id,label
0,4341,"[1, 0, 0, 1, 0, 0, 0]"
1,4136,"[0, 1, 0, 0, 0, 0, 0]"
2,10352,"[1, 0, 0, 0, 0, 1, 0]"
3,8279,"[0, 0, 0, 1, 0, 0, 0]"
4,1164,"[1, 0, 0, 1, 1, 1, 0]"


In [29]:
train_df = train_ids.merge(base, on="row_id", how="left")
dev_df   = dev_ids.merge(base, on="row_id", how="left")

print("Train merged:", train_df.shape, "Missing text:", train_df["paragraph"].isna().sum())
print("Dev merged:", dev_df.shape, "Missing text:", dev_df["paragraph"].isna().sum())

train_df = train_df.dropna(subset=["paragraph"]).copy()
dev_df   = dev_df.dropna(subset=["paragraph"]).copy()

train_df.head()

Train merged: (8375, 5) Missing text: 0
Dev merged: (2094, 5) Missing text: 0


,row_id,label,par_id,par_id_noat,paragraph
0,4341,"[1, 0, 0, 1, 0, 0, 0]",@@17139403,17139403,"The scheme saw an estimated 150,000 children f..."
1,4136,"[0, 1, 0, 0, 0, 0, 0]",@@22273328,22273328,Durban 's homeless communities reconciliation ...
2,10352,"[1, 0, 0, 0, 0, 1, 0]",@@21102155,21102155,The next immediate problem that cropped up was...
3,8279,"[0, 0, 0, 1, 0, 0, 0]",@@21220476,21220476,Far more important than the implications for t...
4,1164,"[1, 0, 0, 1, 1, 1, 0]",@@14727121,14727121,To strengthen child-sensitive social protectio...


In [32]:
# label is a string like "[1, 0, 0, 1, 0, 0, 0]" so literal_eval works
train_df["label_vec"] = train_df["label"].apply(ast.literal_eval)
dev_df["label_vec"]   = dev_df["label"].apply(ast.literal_eval)

# Binary mapping: PCL=1 if any of the 7 category labels is 1
train_df["label_bin"] = train_df["label_vec"].apply(lambda v: int(sum(v) > 0))
dev_df["label_bin"]   = dev_df["label_vec"].apply(lambda v: int(sum(v) > 0))

print("Train binary distribution:\n", train_df["label_bin"].value_counts(normalize=True))
print("Dev binary distribution:\n", dev_df["label_bin"].value_counts(normalize=True))

# Keep only necessary columns
train_df = train_df[["row_id", "par_id", "paragraph", "label_bin"]].copy()
dev_df   = dev_df[["row_id", "par_id", "paragraph", "label_bin"]].copy()

train_df.head()

Train binary distribution:
 label_bin
0    0.905194
1    0.094806
Name: proportion, dtype: float64
Dev binary distribution:
 label_bin
0    0.904967
1    0.095033
Name: proportion, dtype: float64


,row_id,par_id,paragraph,label_bin
0,4341,@@17139403,"The scheme saw an estimated 150,000 children f...",1
1,4136,@@22273328,Durban 's homeless communities reconciliation ...,1
2,10352,@@21102155,The next immediate problem that cropped up was...,1
3,8279,@@21220476,Far more important than the implications for t...,1
4,1164,@@14727121,To strengthen child-sensitive social protectio...,1


In [33]:
# Preprocessing
def clean_text(text: str) -> str:
    text = re.sub(r"<.*?>", "", str(text))   # remove HTML tags
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["text"] = train_df["paragraph"].apply(clean_text)
dev_df["text"]   = dev_df["paragraph"].apply(clean_text)

In [34]:
MODEL_NAME = "roberta-base"
MAX_LEN = 128  # informed by your EDA

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = TextDataset(train_df["text"], train_df["label_bin"])
dev_ds   = TextDataset(dev_df["text"], dev_df["label_bin"])

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [36]:
set_seed(42)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)

    f1 = f1_score(labels, preds, pos_label=1)
    p  = precision_score(labels, preds, pos_label=1, zero_division=0)
    r  = recall_score(labels, preds, pos_label=1, zero_division=0)
    return {"f1_pos": f1, "precision_pos": p, "recall_pos": r}

training_args = TrainingArguments(
    output_dir="../BaselineModel/checkpoints",
    eval_strategy="epoch",          # <-- key change for older transformers
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/pk/zxrx27y90nqcwc591t415lyh0000gp/T/ipykernel_68677/465849261.py:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


In [ ]:
# Dev evaluation
pred = trainer.predict(dev_ds)
dev_logits = pred.predictions
dev_labels = pred.label_ids

dev_probs = torch.softmax(torch.tensor(dev_logits), dim=-1)[:, 1].numpy()
dev_pred = (dev_probs >= 0.5).astype(int)

print("Dev F1 (pos):", f1_score(dev_labels, dev_pred, pos_label=1))
print("Dev Precision (pos):", precision_score(dev_labels, dev_pred, pos_label=1, zero_division=0))
print("Dev Recall (pos):", recall_score(dev_labels, dev_pred, pos_label=1, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(dev_labels, dev_pred))

DEV_OUT = "../BestModel/dev.txt"
with open(DEV_OUT, "w") as f:
    for y in dev_pred:
        f.write(f"{int(y)}\n")

print("Wrote:", DEV_OUT, "Lines:", len(dev_pred))

In [ ]:
# Official test set predictions
test_rows = []
test_bad = 0

with open(TEST_TSV, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        if len(parts) == 5:
            test_rows.append(parts)
        else:
            test_bad += 1

print("Test loaded rows:", len(test_rows), "Bad lines skipped:", test_bad)

test_df = pd.DataFrame(test_rows, columns=["row_id", "par_id", "keyword", "country", "paragraph"])
test_df["text"] = test_df["paragraph"].apply(clean_text)

test_ds = TextDataset(test_df["text"], labels=None)

test_logits = trainer.predict(test_ds).predictions
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1)[:, 1].numpy()
test_pred = (test_probs >= 0.5).astype(int)

TEST_OUT = "../BestModel/test.txt"
with open(TEST_OUT, "w") as f:
    for y in test_pred:
        f.write(f"{int(y)}\n")

print("Wrote:", TEST_OUT, "Lines:", len(test_pred))